## What are Agent Skills?

Agent Skills are a lightweight, open format for extending AI agent capabilities with specialized knowledge and workflows.        

### Skill structure
A skill is a directory containing a SKILL.md file with optional subdirectories for resources:

```text
expense-report/
├── SKILL.md                          # Required - frontmatter + instructions
├── scripts/
│   └── validate.py                   # Executable code agents can run
├── references/
│   └── POLICY_FAQ.md                 # Reference documents loaded on demand
└── assets/
    └── expense-report-template.md    # Templates and static resources
```

### SKILL.md format
The SKILL.md file must contain YAML frontmatter followed by markdown content:
```yaml
name: expense-report
description: File and validate employee expense reports according to company policy. Use when asked about expense submissions, reimbursement rules, or spending limits.
license: Apache-2.0
compatibility: Requires python3
metadata:
  author: contoso-finance
  version: "2.1"
```

Keep SKILL.md under 500 lines and move detailed reference material to separate files.


## Why Agent Skills?

Agents are increasingly capable, but often don't have the context they need to do real work reliably. Skills solve this by packaging procedural knowledge and company-, team-, and user-specific context into portable, version-controlled folders that agents load on demand. This gives agents:

**Domain expertise**: Capture specialized knowledge — from legal review processes to data analysis pipelines to presentation formatting — as reusable instructions and resources.       
**Repeatable workflows**: Turn multi-step tasks into consistent, auditable procedures.               
**Cross-product reuse**: Build a skill once and use it across any skills-compatible agent.          

## How do Agent Skills work?

Agent Skills use a four-stage progressive disclosure pattern to minimize context usage:

**Advertise** (~100 tokens per skill) - Skill names and descriptions are injected into the system prompt at the start of each run, so the agent knows what skills are available.            
**Load** (< 5000 tokens recommended) - When a task matches a skill's domain, the agent calls the load_skill tool to retrieve the full SKILL.md body with detailed instructions.         
**Read resources** (as needed) - The agent calls the read_skill_resource tool to fetch supplementary files (references, templates, assets) only when required.          
**Run scripts** (as needed) - The agent calls the run_skill_script tool to execute scripts bundled with a skill.            
This pattern keeps the agent's context window lean while giving it access to deep domain knowledge on demand.

## Providing skills to an agent

Working with skills involves three building blocks:

**Provider** - SkillsProvider (Python) is a context provider that exposes skills to an agent. It advertises the available skills in the system prompt and registers the tools the agent uses to load skills, read resources, and run scripts.

**Sources** - a source supplies skills to the provider. Skills can come from several source types:
- File-based - skills discovered from SKILL.md files in filesystem directories.
- Code-defined - skills defined inline in code using AgentInlineSkill (C#) or InlineSkill (Python).
- Class-based - skills encapsulated in a class deriving from AgentClassSkill<T> (C#) or ClassSkill (Python).
- MCP-based - skills discovered from MCP (Model Context Protocol) servers via UseMcpSkills (C#) or MCPSkillsSource (Python).            
**Builder** - AgentSkillsProviderBuilder (C#) assembles multiple sources into a single provider, applying aggregation, deduplication, caching, and optional filtering. In Python, compose source classes such as AggregatingSkillsSource, FilteringSkillsSource, and DeduplicatingSkillsSource directly.

The following sections show how to create skills of each source type, then how to combine sources and construct a provider from them.

## Skill sources [ not need]

A SkillsProvider retrieves skills from one or more sources - objects that derive from SkillsSource. Sources fall into two categories: leaf sources that discover or hold skills (such as FileSkillsSource for file-based skills), and decorators that transform the output of another source (aggregation, deduplication, caching, and filtering). You can also create a custom source.

## Provider construction [not need]

SkillsProvider is the component that exposes skills to an agent. It wraps one or more sources and registers the load_skill, read_skill_resource, and run_skill_script tools. There are three ways to create one:

- From skill instances - pass a single Skill or a sequence of skills to the constructor. Best for code-defined and class-based skills. Automatically applies deduplication and caching.
- From file paths - use the SkillsProvider.from_paths() factory. Best for single-source file-based skills. Automatically applies deduplication and caching.
- Direct source composition - construct the source pipeline yourself using the public SkillsSource classes and pass it to the constructor. You control the full pipeline. Best when you need control over ordering, conditional logic, caching keys, or custom decorator behavior.

## Skill filtering 

Use FilteringSkillsSource to control which skills the agent sees. The predicate receives each Skill and the SkillsSourceContext, and returns True to include the skill. For example, to load skills from a shared directory but hide an experimental one

## Tool approval

All tools exposed by SkillsProvider (load_skill, read_skill_resource, and run_skill_script) require approval by default. When a tool call requires approval, the agent pauses and returns approval requests via result.user_input_requests instead of executing immediately. You approve or reject each request with request.to_function_approval_response(approved=...) and send the responses back

## Auto-approving trusted tools

Rather than prompting for every call, install ToolApprovalMiddleware with one of the static auto-approval rules exposed by SkillsProvider. This lets the read-only tools run automatically while still prompting for script execution

## Disabling approval for specific tools

For trusted skills, pass disable_load_skill_approval, disable_read_skill_resource_approval, and/or disable_run_skill_script_approval to opt individual tools out of the approval flow entirely (they are registered with approval_mode="never_require")

## Security best practices

Agent Skills should be treated like any third-party code you bring into your project.Because skill instructions are injected into the agent's context - and skills can include scripts - applying the same level of review and governance you would to an open-source dependency is essential.

- **Review before use** - Read all skill content (SKILL.md, scripts, and resources) before deploying. Verify that a script's actual behavior matches its stated intent. Check for adversarial instructions that attempt to bypass safety guidelines, exfiltrate data, or modify agent configuration files.
- **Source trust** - Only install skills from trusted authors or vetted internal contributors. Prefer skills with clear provenance, version control, and active maintenance. Watch for typosquatted skill names that mimic popular packages.
- **Sandboxing** - Run skills that include executable scripts in isolated environments. Limit filesystem, network, and system-level access to only what the skill requires. Require explicit user confirmation before executing potentially sensitive operations.
- **Audit and logging** - Record which skills are loaded, which resources are read, and which scripts are executed. This gives you an audit trail to trace agent behavior back to specific skill content if something goes wrong.


## When to use skills vs. workflows

Agent Skills and Agent Framework Workflows both extend what agents can do, but they work in fundamentally different ways. Choose the approach that best matches your requirements:

**Control** - With a skill, the AI decides how to execute the instructions. This is ideal when you want the agent to be creative or adaptive. With a workflow, you explicitly define the execution path. Use workflows when you need deterministic, predictable behavior.       
**Resilience** - A skill runs within a single agent turn. If something fails, the entire operation must be retried. Workflows support checkpointing, so they can resume from the last successful step after a failure. Choose workflows when the cost of re-executing the entire process is high.       
**Side effects** - Skills are suitable when operations are idempotent or low-risk. Prefer workflows when steps produce side effects (sending emails, charging payments) that should not be repeated on retry.               
**Complexity** - Skills are best for focused, single-domain tasks that one agent can handle. Workflows are better suited for multi-step business processes that coordinate multiple agents, human approvals, or external system integrations.

In [1]:
print("heelo")

heelo
